# detach-stop-gradient-trick — faded example 1: Complete the discriminator step so G receives no gradient

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `detach-stop-gradient-trick`. Running the beacon reports progress on the `GAN: detach stop-gradient trick` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `GAN: detach stop-gradient trick` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`detach-stop-gradient-trick`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "detach-stop-gradient-trick"
DD_SUBTOPIC = "GAN: detach stop-gradient trick"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

During the discriminator update you must stop gradient from flowing into the generator. The fix is to detach the generator output before scoring it: `G(z).detach()` keeps the same values but removes the autograd graph, so `loss.backward()` only updates D.

## Faded exercise 1

Implement `d_step(G, D, z, x_real)` for the discriminator update. It should compute the fake images from G, score real and fake with D, build `loss = (D(fake) - D(x_real)).mean()`, run `loss.backward()`, and return `loss.item()`. The critical requirement: G's parameters must end with NO gradient. Complete the blanked line that produces the fake tensor.

**Fill in:** Produces the fake images from G's forward pass with the autograd graph severed so backward cannot reach G.

In [ ]:
import torch.nn as nn

t.manual_seed(0)
G = nn.Linear(4, 4)
D = nn.Linear(4, 1)

def d_step(G, D, z, x_real):
    fake = None  # TODO: produce the fake images from G with the autograd graph severed
    loss = (D(fake) - D(x_real)).mean()
    loss.backward()
    return loss.item()


def _test():
    import torch.nn as nn
    t.manual_seed(0)
    Gt = nn.Linear(4, 4)
    Dt = nn.Linear(4, 1)
    zt = t.randn(8, 4)
    xr = t.randn(8, 4)
    for p in list(Gt.parameters()) + list(Dt.parameters()):
        p.grad = None
    lv = d_step(Gt, Dt, zt, xr)
    # G must be untouched by the D-step
    for p in Gt.parameters():
        assert p.grad is None or t.all(p.grad == 0), 'gradient leaked into G'
    # D must have received gradient
    assert all(p.grad is not None for p in Dt.parameters()), 'D got no gradient'
    # loss must match the non-detached forward value
    with t.no_grad():
        ref = (Dt(Gt(zt)) - Dt(xr)).mean().item()
    assert abs(lv - ref) < 1e-5, 'loss value wrong'


try:
    _test()
    _dd_passed.add('faded1')
    print('[Delta Drills] faded1 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import torch.nn as nn

t.manual_seed(0)
G = nn.Linear(4, 4)
D = nn.Linear(4, 1)

def d_step(G, D, z, x_real):
    fake = G(z).detach()
    loss = (D(fake) - D(x_real)).mean()
    loss.backward()
    return loss.item()
```
</details>